In [8]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pypsa

sys.path.append(str(Path.cwd().parent / 'scripts'))
from _tyndp_helpers import _extract_scenario_values, _extract_scenario_values_rowwise

In [5]:
year = 2035
scenario = 'GA'

In [11]:
valpath = Path.cwd().parent / 'data' / 'TYNDP_2024-Scenario-Report-Data-Figures_240522.xlsx'

In [21]:
scenario_mapper = {
    'NT': 'National Trends',
    'DE': 'Distributed Energy',
    'GA': 'Global Ambition',
    'REF': 'Reference',
}

In [6]:
# prenetwork
n = pypsa.Network(Path.cwd().parent / 'resources' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


In [9]:
eu27_countries = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU",
    "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE",
]
eu27_buses = n.buses.index[
    (n.buses.country.isin(eu27_countries)) &
    (n.buses.carrier == 'AC')
]
all_eu27_buses = n.buses.index[
    (n.buses.country.isin(eu27_countries))
]

In [16]:
# adjust EV demand
def final_electricity_transport(path):
    return _extract_scenario_values(path, sheet_name='6-', row_label='Transport')
ev_demand = final_electricity_transport(valpath)

In [17]:
ev_demand

{'National Trends': {2030: 247.64143184472053, 2040: 488.45309616768986},
 'Distributed Energy': {2040: 616.8067207371424, 2050: 849.9178634898052},
 'Global Ambition': {2040: 533.8023438415136, 2050: 779.9316227533446},
 'Reference': {2019: 50.09668955008888}}

In [28]:
weight = (year - 2030) / (2040 - 2030)

tyndp_p_set = ev_demand['National Trends'][2030] * (1 - weight) + ev_demand[scenario_mapper[scenario]][2040] * weight

In [23]:
n.loads.carrier.unique()

array(['electricity', 'land transport EV', 'land transport oil',
       'urban central heat', 'solid biomass for industry',
       'gas for industry', 'H2 for industry', 'industry methanol',
       'naphtha for industry', 'low-temperature heat for industry',
       'industry electricity', 'process emissions', 'NH3',
       'coal for industry', 'shipping methanol', 'shipping oil',
       'kerosene for aviation', 'agriculture electricity',
       'agriculture heat', 'agriculture machinery electric',
       'agriculture machinery oil', 'rural heat', 'urban decentral heat'],
      dtype=object)

In [31]:
tyndp_p_set

390.72188784311703

In [32]:
n = pypsa.Network(Path.cwd().parent / 'resources' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))

loads = n.loads.index[
    (n.loads.carrier == 'land transport EV') &
    (n.loads.index.str.contains('|'.join(eu27_countries)))
]
w = n.snapshot_weightings['generators'].iloc[0]

print('before')
print(n.loads_t.p_set[loads].sum().sum() * w * 1e-6)

factor = tyndp_p_set * 1e6 / (n.loads_t.p_set[loads].sum().sum() * w)
n.loads_t.p_set[loads] *= factor

print('after')
print(n.loads_t.p_set[loads].sum().sum() * w * 1e-6)

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


before
380.99519180496895
after
390.7218878431169


In [34]:
eudh_buses = n.buses.index[
    (n.buses.carrier == 'urban central heat') &
    (n.buses.index.str.contains('|'.join(eu27_countries)))
]

In [35]:
eudh_buses

Index(['AT0 0 urban central heat', 'BE0 0 urban central heat',
       'BG0 0 urban central heat', 'CZ0 0 urban central heat',
       'DE0 0 urban central heat', 'DE0 1 urban central heat',
       'DE0 2 urban central heat', 'DE0 3 urban central heat',
       'DE0 4 urban central heat', 'DE0 5 urban central heat',
       'DK0 0 urban central heat', 'DK1 0 urban central heat',
       'EE0 0 urban central heat', 'ES0 0 urban central heat',
       'ES6 0 urban central heat', 'FI1 0 urban central heat',
       'FR0 0 urban central heat', 'FR0 1 urban central heat',
       'FR0 2 urban central heat', 'FR0 3 urban central heat',
       'FR0 4 urban central heat', 'GR0 0 urban central heat',
       'HR0 0 urban central heat', 'HU0 0 urban central heat',
       'IE3 0 urban central heat', 'IT0 0 urban central heat',
       'IT0 1 urban central heat', 'IT4 0 urban central heat',
       'LT0 0 urban central heat', 'LU0 0 urban central heat',
       'LV0 0 urban central heat', 'NL0 0 urban central

In [ ]:
dh_links = n.links.index[
    (n.links.bus0.isin(eudh_buses)) |
    (n.links.bus1.isin(eudh_buses)) 
]

In [ ]:
n = pypsa.Network(Path.cwd().parent / 'resources' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))